# Experiments on Dan Amrhein's conformal solver (mom6_forge PR #151)

Companion to `04_experiments.md`. Each section reproduces one experiment and saves its
figures to `learn/figures/exp_<name>.png`.

1. **Annulus sector**: a region whose exact conformal grid is known in closed form.
2. **California**: convergence and cost of the test outline, against the finest mesh.
3. **Corner refinement**: #149's refined solver (head `55e7010b`) against #151's.
4. **Crowding**: why Schwarz–Christoffel codes such as gridgen need CRDT, and why the
   finite-element route does not.

Runs on the `dev_mf` env against the `conformal-solver` worktree (head `db0db39`).
The whole notebook takes about 4 minutes on one core.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

WORKTREE = Path(
    "/glade/u/home/manishrv/documents/croc/regional_mom_workflows/mom6_forge-worktrees/conformal-solver"
)
LEARN = Path("/glade/u/home/manishrv/documents/croc/dev/mom6_forge_gridsketcher/learn")
FIG = LEARN / "figures"
FIG.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(WORKTREE))
os.environ.setdefault("ESMFMKFILE", os.path.join(sys.prefix, "lib", "esmf.mk"))
import mom6_forge
import mom6_forge._conformal as cf

assert Path(mom6_forge.__file__).resolve().is_relative_to(WORKTREE), mom6_forge.__file__
head = subprocess.run(["git", "-C", str(WORKTREE), "rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("solver:", cf.__file__, "| worktree HEAD:", head)

# PR #149's solver (head 55e7010b, with corner refinement), written from the worktree's git
# objects to a temporary file and imported by path; the worktree is not touched. It needs
# no other #149 module.
src = subprocess.run(["git", "-C", str(WORKTREE), "show", "55e7010b:mom6_forge/_conformal.py"],
                     capture_output=True, text=True, check=True).stdout
path149 = Path(tempfile.mkdtemp()) / "conformal_pr149.py"
path149.write_text(src)
spec = importlib.util.spec_from_file_location("conformal_pr149", path149)
cf149 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cf149)
print("#149 solver:", path149, "| REFINE =", cf149.REFINE)

RASTER = [2_000, 5_000, 20_000, 80_000, 250_000, 1_000_000]
BLUE, ORANGE, AQUA, VIOLET, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#0b0b0b", "#8a8984"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 140, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df",
                     "grid.linewidth": 0.6, "axes.edgecolor": MUTED, "font.size": 9.5,
                     "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 6})

## Helpers

`measure` runs a solver module's `solve()` unchanged, but wraps its `_solve_mesh` to keep
the mesh, the two fields and their energies. With `pin=(nx, ny)` it overrides the cell
counts the solver would pick, so grids from different meshes line up node for node (the
solver's own choice is kept in `auto_nxny`). Orthogonality uses the same measure as
`tests/test_conformal.py::_ortho` (and #149's `grid_quality`): the deviation from 90° of
the angle between the two grid directions at each cell-corner node, skipping the node
rings within about 2 h of the outline.

In [ ]:
def measure(mod, lon, lat, corners, n_cells, projection, raster_points, pin=None):
    cap, orig = {}, mod._solve_mesh
    if projection is not None and not isinstance(projection, (str, mod.MapProjection)):
        # a MapProjection from the other solver module: rebuild it as this module's class
        projection = mod.MapProjection(projection.kind, projection.lon_0, projection.lat_0,
                                       projection.lat_1, projection.lat_2)

    def spy(*args, **kw):
        f = orig(*args, **kw)
        cap.update(f, auto_nxny=(f["nx"], f["ny"]))
        if pin is not None:
            f["nx"], f["ny"] = pin
        return f

    mod._solve_mesh = spy
    try:
        t0 = time.perf_counter()
        cg = mod.solve(lon, lat, corners, n_cells, projection, raster_points)
        wall = time.perf_counter() - t0
    finally:
        mod._solve_mesh = orig
    K = mod._stiffness(cap["nodes"], cap["tri"])
    e_xi, e_eta = (float(np.nan_to_num(u) @ (K @ np.nan_to_num(u))) for u in (cap["xi"], cap["eta"]))
    return cg, dict(raster=raster_points, nodes=len(cap["nodes"]), h=cap["h"], wall=wall,
                    E_xi=e_xi, E_eta=e_eta, aspect=cap["aspect"], auto_nxny=cap["auto_nxny"],
                    corners=cap["corners"], P=cap["P"])


def node_ortho(qx, qy):
    """Deviation from 90 degrees between grid lines at every node (one-sided at the edges)."""
    ax, ay, bx, by = (np.gradient(q, axis=k) for k in (1, 0) for q in (qx, qy))
    cosang = (ax * bx + ay * by) / (np.hypot(ax, ay) * np.hypot(bx, by))
    return np.abs(90.0 - np.degrees(np.arccos(np.clip(cosang, -1.0, 1.0))))


def trim_rings(cg):
    qx, qy = cg.x[::2, ::2], cg.y[::2, ::2]
    dx = np.median(np.hypot(np.diff(qx, axis=1), np.diff(qy, axis=1)))
    return max(1, int(np.ceil(0.75 + 2.0 * cg.h / dx)))


def ortho_stats(cg):
    n = trim_rings(cg)
    o = node_ortho(cg.x[::2, ::2], cg.y[::2, ::2])[n:-n, n:-n]
    return dict(o50=np.median(o), o99=np.percentile(o, 99), omax=o.max())


def qz(cg):
    """Cell-corner nodes as complex numbers x + iy (projection metres)."""
    return cg.x[::2, ::2] + 1j * cg.y[::2, ::2]


def local_size(z):
    """Local cell size at each node: geometric mean of the two grid steps there."""
    return np.sqrt(np.abs(np.gradient(z, axis=1)) * np.abs(np.gradient(z, axis=0)))


def displacement(cg, z_ref):
    """Distance of each cell-corner node from the reference node, in local cell sizes."""
    d = np.abs(qz(cg) - z_ref) / local_size(z_ref)
    return dict(dmax=d.max(), drms=np.sqrt(np.mean(d**2)))


def folds(cg):
    """Supergrid quads with non-positive signed area (diagonal cross product)."""
    x, y = cg.x, cg.y
    ax, ay = x[1:, 1:] - x[:-1, :-1], y[1:, 1:] - y[:-1, :-1]
    return int(((ax * (y[1:, :-1] - y[:-1, 1:]) - ay * (x[1:, :-1] - x[:-1, 1:])) <= 0).sum())


def show(rows, cols, fmt):
    print("  ".join(f"{c:>10}" for c in cols))
    for r in rows:
        print("  ".join(f"{format(r[c], fmt.get(c, '')):>10}" for c in cols))

## Experiment 1: an exact answer, the annulus sector

The region $r_1<r<r_2$, $\theta_E<\theta<\theta_W$ around a centre $c$ in a fixed
Lambert-conformal plane. $w=\log(z-c)=\ln r+i\theta$ maps it onto a rectangle, so the
exact fields are $\xi=(\theta_W-\theta)/\Delta\theta$ (0 on the west radial side, 1 on
the east) and $\eta=\ln(r/r_1)/\ln(r_2/r_1)$ (0 on the inner arc = south, 1 on the outer
arc = north). Exact energies: $E_\xi=\ln(r_2/r_1)/\Delta\theta$, $E_\eta=1/E_\xi$, so
the modulus in Dan's orientation (aspect $=n_y/n_x$) is $M=\ln(r_2/r_1)/\Delta\theta$.
The exact grid: radial lines at uniform angle, circles at geometrically spaced radii.

In [ ]:
PLANE = cf.MapProjection("lcc", -100.0, 35.0, 33.0, 37.0)
R1, R2, TH_W, TH_E, CENTRE = 300e3, 900e3, np.radians(130.0), np.radians(40.0), -556e3j
# (tilted 5 deg so the solver's "south-west = smallest x + y" rule picks the inner west vertex)
N_ARC = 1024  # chords per arc: sagitta under 0.3 m


def sector(r1, r2, th_w, th_e, centre, n_arc=N_ARC):
    """Outline (lon, lat) and corners: inner arc W -> E, then outer arc E -> W."""
    th = np.linspace(th_w, th_e, n_arc + 1)
    z = np.r_[centre + r1 * np.exp(1j * th), centre + r2 * np.exp(1j * th[::-1])]
    lon, lat = PLANE.to_lonlat(z.real, z.imag)
    return np.asarray(lon), np.asarray(lat), [0, n_arc, n_arc + 1, 2 * n_arc + 1]


def sector_exact(r1, r2, th_w, th_e, centre, nx, ny):
    """Exact cell-corner nodes, shape (ny + 1, nx + 1)."""
    th = th_w - np.linspace(0, 1, nx + 1) * (th_w - th_e)
    r = r1 * (r2 / r1) ** np.linspace(0, 1, ny + 1)
    return centre + r[:, None] * np.exp(1j * th[None, :])


M_EXACT = np.log(R2 / R1) / (TH_W - TH_E)
PIN1 = (80, 56)  # nx, ny for square cells: sqrt(4480 / M) = 80.0, 80 M = 55.95
z_exact = sector_exact(R1, R2, TH_W, TH_E, CENTRE, *PIN1)
print(f"M = ln(r2/r1)/dtheta = {M_EXACT:.6f}  (reciprocal {1 / M_EXACT:.6f})")
print("exact grid orthogonality at interior nodes (should be 0):",
      node_ortho(z_exact.real, z_exact.imag)[1:-1, 1:-1].max())

ann = sector(R1, R2, TH_W, TH_E, CENTRE)
exp1 = []
for rp in RASTER:
    cg, m = measure(cf, *ann, 4480, PLANE, rp, pin=PIN1)
    assert np.allclose([cg.x[0, 0], cg.y[0, 0]], [z_exact[0, 0].real, z_exact[0, 0].imag], atol=1e-3)
    m.update(**ortho_stats(cg), **displacement(cg, z_exact), merr=m["aspect"] / M_EXACT - 1,
             Eerr=m["E_xi"] * m["E_eta"] - 1, Exi_err=m["E_xi"] / M_EXACT - 1,
             Eeta_err=m["E_eta"] * M_EXACT - 1, nxny=f"{m['auto_nxny'][0]}x{m['auto_nxny'][1]}")
    exp1.append(m)
show(exp1, ["raster", "nodes", "h", "nxny", "merr", "Exi_err", "Eeta_err", "Eerr", "dmax", "drms",
            "o50", "o99", "omax", "wall"],
     dict(h=".0f", merr=".2e", Exi_err=".2e", Eeta_err=".2e", Eerr=".2e", dmax=".2e",
          drms=".2e", o50=".4f", o99=".4f", omax=".4f", wall=".2f"))

How much of that is the polygon rather than the solver? Re-run the 250k mesh with arcs
cut into 4x as many chords (sagitta 16x smaller).

In [ ]:
ann_fine = sector(R1, R2, TH_W, TH_E, CENTRE, n_arc=4 * N_ARC)
for rp in (250_000,):
    cg, m = measure(cf, *ann_fine, 4480, PLANE, rp, pin=PIN1)
    print(f"{rp:>9}: merr {m['aspect'] / M_EXACT - 1:+.2e}  E-1 {m['E_xi'] * m['E_eta'] - 1:.2e}  "
          f"dmax {displacement(cg, z_exact)['dmax']:.2e}")

In [ ]:
def slope(xs, ys):
    return np.polyfit(np.log(xs), np.log(ys), 1)[0]


rp = np.array([m["raster"] for m in exp1])
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
ax = axs[0]
for key, lab, c, mk in [("merr", "|relative modulus error|", BLUE, "o"),
                        ("Eerr", r"$E_\xi E_\eta - 1$", ORANGE, "s")]:
    ys = np.abs([m[key] for m in exp1])
    ax.loglog(rp, ys, marker=mk, color=c, label=f"{lab} (slope {slope(rp[2:], ys[2:]):.2f})")
ax.loglog(rp, 0.5 * exp1[0]["Eerr"] * (rp / rp[0]) ** -1.0, ":", color=MUTED, lw=1.2, label=r"$\propto N^{-1} = h^2$")
ax.set(xlabel="raster_points N", ylabel="error", title="Modulus and energy product")
ax.legend(fontsize=8.5)
ax = axs[1]
for key, lab, c, mk in [("dmax", "max node displacement", BLUE, "o"),
                        ("drms", "RMS node displacement", AQUA, "^"),
                        ("omax", "max orthogonality error (deg)", ORANGE, "s"),
                        ("o99", "p99 orthogonality error (deg)", VIOLET, "D")]:
    ys = np.array([m[key] for m in exp1])
    ax.loglog(rp, ys, marker=mk, color=c, label=f"{lab} (slope {slope(rp[2:], ys[2:]):.2f})")
ax.set(xlabel="raster_points N", ylabel="cells  /  degrees", title="Grid nodes vs the exact grid (80 x 56 cells)")
ax.legend(fontsize=8.5)
fig.suptitle("Annulus sector, r2/r1 = 3, 90 deg: #151 converges to the exact map", fontsize=11)
fig.tight_layout()
fig.savefig(FIG / "exp_annulus_convergence.png", bbox_inches="tight")

In [ ]:
cg, _ = measure(cf, *ann, 4480, PLANE, 20_000, pin=PIN1)
d = np.abs(qz(cg) - z_exact) / local_size(z_exact)
fig, axs = plt.subplots(1, 2, figsize=(10, 4.2))
ax = axs[0]
q = qz(cg)
for j in range(0, q.shape[0], 4):
    ax.plot(q[j].real / 1e3, q[j].imag / 1e3, color=BLUE, lw=0.6)
for i in range(0, q.shape[1], 4):
    ax.plot(q[:, i].real / 1e3, q[:, i].imag / 1e3, color=BLUE, lw=0.6)
ax.plot(z_exact[::4, ::4].real.ravel() / 1e3, z_exact[::4, ::4].imag.ravel() / 1e3, ".", color=ORANGE, ms=3)
ax.set(aspect="equal", xlabel="x (km)", ylabel="y (km)",
       title="Every 4th grid line, 20k mesh (blue);\nexact nodes (orange)")
ax = axs[1]
pm = ax.pcolormesh(q.real / 1e3, q.imag / 1e3, d, cmap="Blues", shading="gouraud")
fig.colorbar(pm, ax=ax, shrink=0.75, label="node displacement (cells)")
ax.set(aspect="equal", xlabel="x (km)", title="Distance from the exact node, 20k mesh")
fig.tight_layout()
fig.savefig(FIG / "exp_annulus_grid.png", bbox_inches="tight")

### The 1000 x 600 km rectangle (as in the test fixture) is reproduced exactly

In [ ]:
rx, ry = [-5e5, 5e5, 5e5, -5e5], [-3e5, -3e5, 3e5, 3e5]
rlon, rlat = (np.asarray(a) for a in PLANE.to_lonlat(rx, ry))
qx_ex, qy_ex = np.meshgrid(-5e5 + 4e4 * np.arange(26), -3e5 + 4e4 * np.arange(16))
for rp in (2_000, 20_000, 250_000):
    cg, m = measure(cf, rlon, rlat, [0, 1, 2, 3], 375, PLANE, rp)
    dm = np.abs(qz(cg) - (qx_ex + 1j * qy_ex)).max()
    print(f"{rp:>8}: {cg.nx}x{cg.ny}  aspect-0.6 {m['aspect'] - 0.6:+.1e}  E-1 {m['E_xi'] * m['E_eta'] - 1:+.1e}  "
          f"max node error {dm:.1e} m  max ortho {ortho_stats(cg)['omax']:.1e} deg")

## Experiment 2: California, convergence and cost

Same quantities as experiment 1, against the 1M-point mesh. The cell counts are pinned
to the 1M mesh's choice so the grids line up node for node. There is no exact modulus,
so the modulus error is measured against #149's refined solver at 1M points, whose value
has stopped changing (experiment 3 shows why it is the better reference).

In [ ]:
CAL = json.loads((WORKTREE / "tests/data/california_bays.outline.json").read_text())["outline"]
cal = (CAL["lon"], CAL["lat"], CAL["corners"])
CAL_PLANE = cf.as_projection(None, CAL["lon"], CAL["lat"])
print(CAL_PLANE)

cal_ref, ref = measure(cf, *cal, 12_000, CAL_PLANE, 1_000_000)
PIN2 = ref["auto_nxny"]
z_ref = qz(cal_ref)
ref149_cg, ref149 = measure(cf149, *cal, 12_000, CAL_PLANE, 1_000_000, pin=PIN2)
M_BEST = ref149["aspect"]
print(f"best modulus estimate (#149, 1M): {M_BEST:.6f}; 63 M = {63 * M_BEST:.3f}")
exp2 = []
for rp in RASTER:
    cg, m = (cal_ref, ref) if rp == 1_000_000 else measure(cf, *cal, 12_000, CAL_PLANE, rp, pin=PIN2)
    m = dict(m)
    m.update(**ortho_stats(cg), **displacement(cg, z_ref), Eerr=m["E_xi"] * m["E_eta"] - 1,
             nxny=f"{m['auto_nxny'][0]}x{m['auto_nxny'][1]}", merr=m["aspect"] / M_BEST - 1,
             bound=0.5 * np.log(m["E_xi"] * m["E_eta"]), ny63=63 * m["aspect"])
    exp2.append(m)
show(exp2, ["raster", "nodes", "h", "nxny", "ny63", "aspect", "merr", "bound", "Eerr", "dmax", "drms", "o50", "o99",
            "omax", "wall"],
     dict(h=".0f", ny63=".3f", aspect=".6f", merr=".1e", bound=".1e", Eerr=".2e", dmax=".3f", drms=".4f",
          o50=".3f", o99=".2f", omax=".1f", wall=".2f"))
# where the largest displacement is
for m_ in exp2[3:5]:
    cg = measure(cf, *cal, 12_000, CAL_PLANE, m_["raster"], pin=PIN2)[0]
    dd = np.abs(qz(cg) - z_ref) / local_size(z_ref)
    j, i = np.unravel_index(np.argmax(dd), dd.shape)
    v = int(np.argmin(np.abs(ref["P"][:, 0] + 1j * ref["P"][:, 1] - z_ref[j, i])))
    print(f"{m_['raster']:>7}: max displacement {dd.max():.2f} cells at node ({j}, {i}), "
          f"{np.abs(ref['P'][v, 0] + 1j * ref['P'][v, 1] - z_ref[j, i]) / 1e3:.1f} km from vertex {v}; "
          f"{(dd > 0.5).sum()} of {dd.size} nodes moved more than half a cell")

The singularities that limit convergence. A vertex of interior angle $\alpha$ makes the
fields behave like $r^{\lambda}$ with $\lambda=\pi/\alpha$ on a side, and
$\lambda=\pi/(2\alpha)$ at a grid corner (where the boundary condition switches between
fixed value and zero flux). On a uniform mesh the energy error goes like
$h^{2\lambda_{\min}} = N^{-\lambda_{\min}}$.

In [ ]:
ang = cf.interior_angles(*ref["P"].T)
lam = 180.0 / ang
lam[ref["corners"]] = 90.0 / ang[ref["corners"]]
for i in np.argsort(lam)[:5]:
    kind = "corner" if i in ref["corners"] else "side vertex"
    print(f"vertex {i:2d} ({kind:11s}): angle {ang[i]:6.1f} deg -> lambda = {lam[i]:.3f}")
rp = np.array(RASTER)
Eerr = np.array([m["Eerr"] for m in exp2])
print(f"fitted slope of E_xi E_eta - 1 vs N (all points): {slope(rp, Eerr):.3f}; theory -{lam.min():.3f}")
merr = np.abs([m["merr"] for m in exp2])
print("the bound |ln(aspect / M)| <= ln(E_xi E_eta) / 2 holds at every mesh:",
      all(abs(np.log(1 + m["merr"])) <= m["bound"] for m in exp2))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
ax = axs[0]
ax.loglog(rp, Eerr, "s-", color=ORANGE, label=rf"$E_\xi E_\eta-1$ (slope {slope(rp, Eerr):.2f})")
ax.loglog(rp, merr, "o-", color=BLUE, label="|modulus error| (vs #149 at 1M)")
ax.loglog(rp, Eerr[0] * (rp / rp[0]) ** -lam.min(), ":", color=MUTED, lw=1.2,
          label=rf"$\propto N^{{-{lam.min():.2f}}}$ (203.5° corner)")
ax.set(xlabel="raster_points N", title="Modulus error")
ax.legend(fontsize=8)
ax = axs[1]
for key, lab, c, mk in [("dmax", "max node displacement (cells)", BLUE, "o"),
                        ("drms", "RMS node displacement (cells)", AQUA, "^")]:
    ys = np.array([m[key] for m in exp2[:-1]])
    ax.loglog(rp[:-1], ys, marker=mk, color=c, label=f"{lab} (slope {slope(rp[:-1], ys):.2f})")
ax.set(xlabel="raster_points N", title="Nodes vs #151's own 1M-point grid (63 x 192)")
ax.legend(fontsize=8)
ax = axs[2]
wall = np.array([m["wall"] for m in exp2])
ax.loglog(rp, wall, "o-", color=VIOLET, label=f"solve() wall time (slope {slope(rp[2:], wall[2:]):.2f})")
ax.loglog(rp, wall[-1] * (rp / rp[-1]), ":", color=MUTED, lw=1.2, label=r"$\propto N$")
ax.set(xlabel="raster_points N", ylabel="seconds (one core)", title="Cost")
ax.legend(fontsize=8)
fig.suptitle("California outline, #151 (no refinement), 12,000 cells", fontsize=11)
fig.tight_layout()
fig.savefig(FIG / "exp_california_convergence.png", bbox_inches="tight")

Cost also grows with the number of outline vertices: `_mesh` measures the distance from
every lattice point to the outline ring (`shapely.distance`, no spatial index) and tests
containment against the whole path. The annulus sector at 250k points, arcs cut into
more and more chords:

In [ ]:
for n_arc in (64, 256, 1024, 4096):
    _, m = measure(cf, *sector(R1, R2, TH_W, TH_E, CENTRE, n_arc), 4480, PLANE, 250_000)
    print(f"{2 * n_arc + 2:>5} vertices: {m['wall']:5.2f} s")

## Experiment 3: corner refinement, #149 (`55e7010b`) against #151

`cf149` (loaded in the setup cell) refines the mesh round the sharpest singular vertices:
`raster_points // 4000` of them (corners over 100°, side vertices over 200°), halving the
spacing 5 times within about 6 h.

In [ ]:
print("REFINE =", cf149.REFINE, "| PREVIEW_RASTER_POINTS =", cf149.PREVIEW_RASTER_POINTS)
P = ref["P"]
for rp_ in (20_000, 250_000):
    spots = cf149._spots(P, ref["corners"], rp_ // 4000)
    idx = [int(np.argmin(np.hypot(*(P - s).T))) for s in spots]
    print(f"{rp_:>7}: {len(spots)} refined vertices {idx}")

In [ ]:
exp3, grids3 = [], {}
for rp_ in (20_000, 250_000):
    for name, mod in (("#151", cf), ("#149", cf149)):
        cg, m = measure(mod, *cal, 12_000, CAL_PLANE, rp_)
        m.update(**ortho_stats(cg), name=name, folds=folds(cg), Eerr=m["E_xi"] * m["E_eta"] - 1,
                 nxny=f"{cg.nx}x{cg.ny}")
        exp3.append(m)
        grids3[name, rp_] = cg
show(exp3, ["name", "raster", "nodes", "nxny", "aspect", "Eerr", "o50", "o99", "omax", "folds", "wall"],
     dict(aspect=".6f", Eerr=".2e", o50=".3f", o99=".2f", omax=".1f", wall=".2f"))

Convergence of the modulus with refinement, and node displacement against a common
reference (#149 at 1M points, cell counts pinned to 63 x 192).

In [ ]:
z149 = qz(ref149_cg)  # computed in experiment 2
exp3b = []
for rp_ in RASTER:
    for name, mod in (("#151", cf), ("#149", cf149)):
        if (name, rp_) == ("#149", 1_000_000):
            cg, m = ref149_cg, dict(ref149)
        elif (name, rp_) == ("#151", 1_000_000):
            cg, m = cal_ref, dict(ref)
        else:
            cg, m = measure(mod, *cal, 12_000, CAL_PLANE, rp_, pin=PIN2)
        m.update(**displacement(cg, z149), **ortho_stats(cg), name=name, Eerr=m["E_xi"] * m["E_eta"] - 1)
        exp3b.append(m)
show(exp3b, ["name", "raster", "nodes", "aspect", "Eerr", "dmax", "drms", "o99", "omax", "wall"],
     dict(aspect=".6f", Eerr=".2e", dmax=".3f", drms=".4f", o99=".2f", omax=".1f", wall=".2f"))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for name, c, mk in (("#151", BLUE, "o"), ("#149", ORANGE, "s")):
    rows = [m for m in exp3b if m["name"] == name]
    n = np.array([m["nodes"] for m in rows])
    axs[0].loglog(n, [m["Eerr"] for m in rows], marker=mk, color=c, label=name)
    axs[1].loglog(n, [m["omax"] for m in rows], marker=mk, color=c, label=f"{name} max")
    axs[1].loglog(n, [m["o99"] for m in rows], marker=mk, color=c, ls="--", mfc="white", label=f"{name} p99")
    rows = [m for m in rows if m["raster"] < 1_000_000] if name == "#149" else rows
    axs[2].loglog([m["nodes"] for m in rows], [m["dmax"] for m in rows], marker=mk, color=c, label=f"{name} max")
    axs[2].loglog([m["nodes"] for m in rows], [m["drms"] for m in rows], marker=mk, color=c, ls="--",
                  mfc="white", label=f"{name} RMS")
axs[0].set(xlabel="mesh nodes", title=r"$E_\xi E_\eta - 1$ (modulus error gauge)")
axs[1].set(xlabel="mesh nodes", ylabel="degrees", title="Orthogonality error (interior nodes)")
axs[2].set(xlabel="mesh nodes", ylabel="cells", title="Node displacement vs #149 at 1M")
for ax in axs:
    ax.legend(fontsize=8)
fig.suptitle("California: corner refinement (#149) vs uniform mesh (#151)", fontsize=11)
fig.tight_layout()
fig.savefig(FIG / "exp_refinement_convergence.png", bbox_inches="tight")

#149's p99 and max stop falling from about 80k points on. Is what is left solver error,
or the measure itself? Centred differences on a curved grid are not exactly
perpendicular even when the map is exactly conformal, most of all next to singular
vertices. Keep the 1M refined mesh and change only the cell count:

In [ ]:
for nc in (3_000, 12_000, 48_000):
    cg = measure(cf149, *cal, nc, CAL_PLANE, 1_000_000)[0]
    s = ortho_stats(cg)
    print(f"{nc:>6} cells ({cg.nx}x{cg.ny}): p50 {s['o50']:.3f}  p99 {s['o99']:.3f}  max {s['omax']:.2f} deg")

Where the nodes move: distance of each node from #149's 1M grid, 250k meshes.

In [ ]:
P = ref["P"]
fig, axs = plt.subplots(1, 2, figsize=(8.5, 7), sharey=True)
for ax, name, mod in zip(axs, ("#151", "#149"), (cf, cf149)):
    cg = measure(mod, *cal, 12_000, CAL_PLANE, 250_000, pin=PIN2)[0]
    q = qz(cg)
    dd = np.abs(q - z149) / local_size(z149)
    pm = ax.pcolormesh(q.real / 1e3, q.imag / 1e3, dd, cmap="Blues", norm=LogNorm(1e-4, 3), shading="gouraud")
    ax.plot(*np.r_[P, P[:1]].T / 1e3, color=INK, lw=0.7)
    ax.set(aspect="equal", xlabel="x (km)",
           title=f"{name}, 250k mesh\nmax {dd.max():.2f}, RMS {np.sqrt(np.mean(dd**2)):.4f} cells")
axs[0].set_ylabel("y (km)")
fig.colorbar(pm, ax=axs, shrink=0.6, label="node displacement from #149 at 1M (cells)")
fig.savefig(FIG / "exp_refinement_displacement.png", bbox_inches="tight")

Maps of the orthogonality error. Each cell shows the worst of its 4 corner nodes (as
#149's `cell_metrics` does); the dashed line is the edge of the trimmed band the
statistics skip. Labelled dots are the singular vertices (corners over 100°, others over
200°).

In [ ]:
sing = [i for i in range(len(P)) if (ang[i] > 100 if i in ref["corners"] else ang[i] > 200)]


def cell_ortho(cg):
    o = node_ortho(cg.x[::2, ::2], cg.y[::2, ::2])
    return np.maximum.reduce([o[:-1, :-1], o[:-1, 1:], o[1:, :-1], o[1:, 1:]])


def band(cg):
    n, q = trim_rings(cg), qz(cg)
    r = np.r_[q[n, n:-n], q[n:-n, -n - 1], q[-n - 1, -n - 1:n - 1:-1], q[-n - 1:n - 1:-1, n]]
    return r.real / 1e3, r.imag / 1e3


norm = LogNorm(0.01, 30)
fig, axs = plt.subplots(1, 4, figsize=(14, 8.5), sharey=True)
for ax, key in zip(axs, [("#151", 20_000), ("#149", 20_000), ("#151", 250_000), ("#149", 250_000)]):
    cg = grids3[key]
    q = qz(cg)
    pm = ax.pcolormesh(q.real / 1e3, q.imag / 1e3, cell_ortho(cg), cmap="Blues", norm=norm)
    ax.plot(*np.r_[P, P[:1]].T / 1e3, color=INK, lw=0.7)
    ax.plot(*band(cg), "--", color=MUTED, lw=0.7)
    s = ortho_stats(cg)
    ax.set(aspect="equal", xlabel="x (km)",
           title=f"{key[0]}, {key[1] // 1000}k mesh\np99 {s['o99']:.2f}°, max {s['omax']:.1f}°")
    for i in sing:
        ax.plot(*P[i] / 1e3, "o", color=ORANGE, ms=4)
        ax.annotate(str(i), P[i] / 1e3, xytext=(5, -3), textcoords="offset points", fontsize=7.5, color=INK)
axs[0].set_ylabel("y (km)")
fig.colorbar(pm, ax=axs, shrink=0.5, label="orthogonality error (deg), worst corner of each cell")
print("singular vertices:", {i: round(float(ang[i]), 1) for i in sing})
fig.savefig(FIG / "exp_refinement_maps.png", bbox_inches="tight")

Zooms. First, where each grid's worst interior node is; then both solvers' cells (250k
meshes) round the NE corner (vertex 22, 203.5°), the SE corner (vertex 6, 185.7°, with
the 249° headland at vertex 5 next to it) and the 247° headland at vertex 11.

In [ ]:
worst = []
for name in ("#151", "#149"):
    for rp_ in (20_000, 250_000):
        cg = grids3[name, rp_]
        n = trim_rings(cg)
        o = node_ortho(cg.x[::2, ::2], cg.y[::2, ::2])[n:-n, n:-n]
        j, i = np.unravel_index(np.argmax(o), o.shape)
        z = qz(cg)[j + n, i + n]
        v = int(np.argmin(np.abs(P[:, 0] + 1j * P[:, 1] - z)))
        print(f"{name} {rp_ // 1000}k: worst interior node {o.max():.1f} deg at node ({j + n}, {i + n}), "
              f"{np.abs(P[v, 0] + 1j * P[v, 1] - z) / 1e3:.1f} km from vertex {v} ({ang[v]:.1f} deg)")
        worst.append(v)
zoom_vertices = [22, 6, 11]

fig, axs = plt.subplots(len(zoom_vertices), 2, figsize=(9, 4.2 * len(zoom_vertices)), squeeze=False)
for row, v in zip(axs, zoom_vertices):
    for ax, name in zip(row, ("#151", "#149")):
        cg = grids3[name, 250_000]
        q = qz(cg)
        R = 5 * np.median(local_size(q))
        pm = ax.pcolormesh(q.real / 1e3, q.imag / 1e3, cell_ortho(cg), cmap="Blues", norm=norm,
                           edgecolors="#b9b8b2", linewidth=0.3)
        ax.plot(*np.r_[P, P[:1]].T / 1e3, color=INK, lw=1.0)
        ax.plot(*band(cg), "--", color=MUTED, lw=0.8)
        ax.plot(*P[v] / 1e3, "o", color=ORANGE, ms=6)
        ax.set(xlim=((P[v, 0] - R) / 1e3, (P[v, 0] + R) / 1e3), ylim=((P[v, 1] - R) / 1e3, (P[v, 1] + R) / 1e3),
               aspect="equal", title=f"{name}, 250k: vertex {v} ({ang[v]:.1f}°)", xlabel="x (km)", ylabel="y (km)")
        ax.grid(False)
fig.colorbar(pm, ax=axs, shrink=0.5, label="orthogonality error (deg)")
fig.savefig(FIG / "exp_refinement_zooms.png", bbox_inches="tight")

The other rows of the ROADMAP table: the reflex ring round the pole (#149's
`test_a_reflex_grid_corner_folds_no_cell`: 40,000 cells, preview mesh) and a plain box.

In [ ]:
ring = (np.linspace(0, 360, 48, endpoint=False), 80 + 9 * np.cos(np.radians(np.linspace(0, 360, 48, endpoint=False))),
        [0, 12, 24, 36])
box = ([-125.0, -120.0, -120.0, -125.0], [35.0, 35.0, 40.0, 40.0], [0, 1, 2, 3])
for label, args, n_cells in (("ring round pole", ring, 40_000), ("lon/lat box 5x5 deg", box, 12_000)):
    for rp_ in (20_000, 250_000):
        out = []
        for name, mod in (("#151", cf), ("#149", cf149)):
            cg, m = measure(mod, *args, n_cells, None, rp_)
            s = ortho_stats(cg)
            out.append(f"{name}: p99 {s['o99']:.2f} max {s['omax']:.2f} folds {folds(cg)}")
        print(f"{label:20s} {rp_ // 1000:>4}k | " + " | ".join(out))

## Experiment 4: crowding, why gridgen's Schwarz–Christoffel route needs CRDT

The Schwarz–Christoffel map from the upper half-plane onto a rectangle is
$f(z)=\int_0^z \frac{dt}{\sqrt{(1-t^2)(1-k^2t^2)}}$, with prevertices $\pm1, \pm1/k$ and
image the rectangle $[-K, K]\times[0, K']$, where $K=K(k)$, $K'=K(\sqrt{1-k^2})$.
Aspect $L = 2K/K'$ (long over short side). Here $m' = 1-k^2$ is found for each $L$
with `ellipkm1`, which keeps full relative accuracy as $m'\to0$.

In [ ]:
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.special import ellipk, ellipkm1


def mprime(L):
    """m' = 1 - k^2 of the rectangle with aspect L = 2K(k)/K(k')."""
    g = lambda s: 2 * ellipkm1(np.exp(s)) / ellipk(np.exp(s)) - L
    return np.exp(brentq(g, np.log(1e-300), np.log(0.999999), xtol=1e-14))


def gaps(L):
    """Smallest prevertex gaps for aspect L, computed without cancellation."""
    mp = mprime(L)
    one_minus_k = -np.expm1(0.5 * np.log1p(-mp))  # 1 - k
    k = 1 - one_minus_k
    return dict(
        line=np.expm1(-0.5 * np.log1p(-mp)),  # 1/k - 1: prevertices +-1, +-1/k
        disk=2 * np.arctan(one_minus_k / (2 * np.sqrt(k))),  # unit disk, centre of rectangle -> 0
        zero_one_inf=one_minus_k**2 / (4 * k),  # 3 prevertices sent to 0, 1, infinity
        cross_ratio=(one_minus_k / (1 + k)) ** 2,  # (x2-x1)(x4-x3)/((x3-x1)(x4-x2))
        mp=mp)


# Check the setup by quadrature at a moderate aspect: side lengths and the centre's preimage
L = 4.0
mp = mprime(L)
k = np.sqrt(1 - mp)
f = lambda t: 1 / np.sqrt((1 - t * t) * (1 - k * k * t * t))
K = quad(f, 0, 1, limit=200)[0]
Kp = quad(lambda t: 1 / np.sqrt((t * t - 1) * (1 - k * k * t * t)), 1, 1 / k, limit=200)[0]
centre = quad(lambda s: 1 / np.sqrt((1 + s * s) * (1 + k * k * s * s)), 0, 1 / np.sqrt(k))[0]
print(f"L = {L}: quadrature 2K/K' = {2 * K / Kp:.10f}; f(i/sqrt k) = i*{centre:.6f}, K'/2 = {Kp / 2:.6f}")

Ls = np.linspace(1, 40, 157)
G = {key: np.array([gaps(L)[key] for L in Ls]) for key in ("line", "disk", "zero_one_inf", "cross_ratio")}
big = Ls >= 6
for key in G:
    rate = -np.polyfit(Ls[big], np.log(G[key][big]), 1)[0]
    pref = np.exp(np.mean(np.log(G[key][big]) + rate * Ls[big]))
    print(f"{key:13s}: gap ~ {pref:5.2f} exp(-{rate / np.pi:.4f} pi L)")
EPS = np.finfo(float).eps
for key in ("disk", "zero_one_inf"):
    print(f"{key}: gap = eps at L = {brentq(lambda L: np.log(gaps(L)[key]) - np.log(EPS), 2, 40):.2f}")

What double precision does to the best-case (disk, symmetric) prevertices: the two
prevertex angles of each short side sit at $\pi/2 \pm \delta/2$. Round them to doubles,
take the difference and invert it back to an aspect ratio.

In [ ]:
def aspect_from_gap(delta):
    if delta <= 0:
        return np.inf
    u = np.tan(delta / 2)
    s = np.sqrt(1 + u * u) - u  # sqrt(k)
    mp = 2 * u * s * (1 + s * s)  # 1 - k^2 = (1 - k)(1 + k), 1 - k = 2 u sqrt(k)
    return 2 * ellipkm1(mp) / ellipk(mp)


Ld = np.linspace(1, 30, 600)
L_back = []
for L in Ld:
    dlt = gaps(L)["disk"]
    a, b = np.float64(np.pi / 2) + dlt / 2, np.float64(np.pi / 2) - dlt / 2
    L_back.append(aspect_from_gap(a - b))
L_back = np.array(L_back)
bad = np.abs(L_back / Ld - 1) > 0.01
print(f"recovered aspect off by >1% from L = {Ld[bad].min():.1f}; infinite from L = {Ld[np.isinf(L_back)].min():.1f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axs[0]
ax.semilogy(Ls, G["disk"], color=BLUE, label="unit disk, centre at rectangle centre (best case)")
ax.semilogy(Ls, G["line"], "--", color=AQUA, label=r"real line, prevertices $\pm1, \pm1/k$")
ax.semilogy(Ls, G["zero_one_inf"], color=ORANGE, label=r"real line, 3 prevertices at $0, 1, \infty$")
ax.semilogy(Ls, 8 * np.exp(-np.pi * Ls / 2), ":", color=MUTED, lw=1.2)
ax.semilogy(Ls, 16 * np.exp(-np.pi * Ls), ":", color=MUTED, lw=1.2)
ax.annotate(r"$8e^{-\pi L/2}$", (34, 8 * np.exp(-np.pi * 17)), xytext=(4, 6), textcoords="offset points", color=MUTED)
ax.annotate(r"$16e^{-\pi L}$", (21, 16 * np.exp(-np.pi * 21)), xytext=(6, 4), textcoords="offset points", color=MUTED)
ax.axhline(EPS, color=INK, lw=1)
for key, c in (("disk", BLUE), ("zero_one_inf", ORANGE)):
    Lx = brentq(lambda L: np.log(gaps(L)[key]) - np.log(EPS), 2, 40)
    ax.plot(Lx, EPS, "o", color=c, ms=7, mec="white", mew=1.5)
    ax.annotate(f"L = {Lx:.1f}", (Lx, EPS), xytext=(6, -14), textcoords="offset points", fontsize=8.5)
ax.annotate("double-precision epsilon", (1.5, EPS), xytext=(0, 4), textcoords="offset points", fontsize=8.5)
ax.set(xlabel="rectangle aspect ratio L (long / short side)", ylabel="smallest gap between prevertices",
       ylim=(1e-40, 10), title="Prevertex crowding for a rectangle")
ax.legend(fontsize=8, loc="lower left")
ax = axs[1]
ax.plot(Ld, Ld, ":", color=MUTED, lw=1.2, label="exact")
ax.plot(Ld, np.where(np.isinf(L_back), np.nan, L_back), color=BLUE,
        label="aspect recovered from\nprevertex angles stored as doubles")
ax.axvline(Ld[np.isinf(L_back)].min(), color=ORANGE, lw=1)
ax.annotate("angles round to the same double:\nthe rectangle cannot be represented", (Ld[np.isinf(L_back)].min(), 6),
            xytext=(-8, 0), textcoords="offset points", ha="right", fontsize=8.5)
ax.set(xlabel="true aspect ratio L", ylabel="aspect ratio the prevertices encode", title="What float64 can represent",
       xlim=(1, 30), ylim=(0, 32))
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(FIG / "exp_crowding.png", bbox_inches="tight")

The finite-element route has no prevertices, so nothing crowds. Thin annulus sectors
(curved channels, 90° of arc) with exact modulus $1/L$, solved by #151 at the build mesh.

In [ ]:
for L in (5, 10, 20, 30, 50):
    r1 = 500e3
    r2 = r1 * np.exp((TH_W - TH_E) / L)
    chan = sector(r1, r2, TH_W, TH_E, -425e3j)
    cg, m = measure(cf, *chan, 30 * L * L, PLANE, 250_000)
    exact = 1 / L
    print(f"L = {L:2d}: channel {(r2 - r1) / 1e3:5.1f} km wide; modulus error {m['aspect'] / exact - 1:+.1e}, "
          f"E-1 {m['E_xi'] * m['E_eta'] - 1:.1e}, {cg.nx}x{cg.ny} cells, ortho p99 {ortho_stats(cg)['o99']:.3f} deg, "
          f"{m['wall']:.1f} s; SC disk gap would be {gaps(L)['disk']:.0e}")